# TODO: For now written by gPT, Singular value decomposition (SVD)

One matrix, from arrows to a simpler approximation. Following Gilbert Strang’s MIT lecture.

## 1 · A vector is an arrow

Its coordinates say how far to move along each axis.

<img src="images/svd/01-vector.png" width="75%" alt="The vector (2,1) means move 2 right and 1 up.">

## 2 · A matrix changes an arrow

Multiply each row by the vector’s coordinates and add. The same matrix works on every input arrow.

<img src="images/svd/02-matrix.png" width="75%" alt="M has rows (3,2) and (-2,-3). Row multiplication maps (2,1) to (8,-7).">

## 3 · Try every direction

Length-1 arrows form a **unit circle**. Their outputs form an ellipse. Here the **singular values**, 5 and 1, are the longest and shortest output lengths. They are nonnegative: below 1 shrinks; zero removes a direction.

<img src="images/svd/03-stretches.png" width="75%" alt="With the same axis scale, a unit circle maps to an ellipse with longest radius 5 and shortest radius 1.">

## 4 · Find the special directions

SVD finds perpendicular input directions whose outputs stay perpendicular. (Perpendicular = 90° apart.) **v** and **u** are unit arrows; **σ** supplies the length.

<img src="images/svd/04-directions.png" width="75%" alt="Perpendicular unit input directions v1,v2 map to 5u1 and 1u2. In general M vi equals sigma i times ui.">

## 5 · Split the matrix into three actions

Every real matrix has an SVD. **V** holds input directions as columns; **U** holds output directions. **ᵀ** swaps rows and columns. For perpendicular unit axes, **Vᵀ** undoes V’s orientation. **Σ** has stretch factors on its diagonal. The rightmost matrix acts first.

<img src="images/svd/05-decomposition.png" width="90%" alt="V transpose aligns the input axes, diagonal Sigma stretches them by 5 and 1, and U reorients them into the output axes. M equals U Sigma V transpose.">

In [1]:
import numpy as np

M = np.array([[3., 2.], [-2., -3.]])
x = np.array([2., 1.])
U, s, Vt = np.linalg.svd(M, full_matrices=False)

aligned = Vt @ x              # @ means matrix multiplication.
stretched = np.diag(s) @ aligned
result = U @ stretched

print("Singular values:", s)
print("Three actions:", result, " | Direct M @ x:", M @ x)
print("Rebuilds M:", np.allclose(M, U @ np.diag(s) @ Vt))

Singular values: [5. 1.]
Three actions: [ 8. -7.]  | Direct M @ x: [ 8. -7.]
Rebuilds M: True


## 6 · Split it into pieces; keep the strongest

Each piece is a **column × a row**. It measures input along one v direction, then sends that amount along its u direction, scaled by σ. Its outputs lie on one line: **rank 1**. Rank counts independent output directions, or nonzero singular values.

<img src="images/svd/06-rank.png" width="85%" alt="The strongest rank-1 piece, with entries 2.5,2.5,-2.5,-2.5, plus the weaker rank-1 piece, with entries 0.5,-0.5,0.5,-0.5, equals the original rank-2 matrix. Keep the first piece to approximate M.">

In [2]:
# np.outer multiplies a column by a row: one rank-1 piece.
M1 = s[0] * np.outer(U[:, 0], Vt[0, :])

print("Keep the strongest piece:\n", M1.round(1))
print("Original:", M @ x, " | Approximation:", M1 @ x)

Keep the strongest piece:
 [[ 2.5  2.5]
 [-2.5 -2.5]]
Original: [ 8. -7.]  | Approximation: [ 7.5 -7.5]


Keep the largest **r** singular values → the closest rank-r matrix (least total squared entry error). SVD also works for rectangular matrices, with different numbers of input and output coordinates.

**LoRA connection:** SVD finds factors of an existing matrix; LoRA learns factors of a weight update.

<small>Sources: [Gilbert Strang / MIT: directions (8:03)](https://www.youtube.com/watch?v=rYz83XPxiZo&t=483s) · [geometry (28:30)](https://www.youtube.com/watch?v=rYz83XPxiZo&t=1710s) · [rank-one pieces (50:16)](https://www.youtube.com/watch?v=rYz83XPxiZo&t=3016s) · [NumPy](https://numpy.org/doc/stable/reference/generated/numpy.linalg.svd.html) · [low-rank approximation](https://cs357.cs.illinois.edu/textbook/notes/svd.html). Diagrams use a simple numerical example.</small>